In [ ]:
import zarr
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import pandas as pd

Loading the data

In [ ]:
# access data (currently mounted on the VM) --> no need to call GCP bucket storage for now
store = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_small.zarr')
store_eve = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_eve.zarr')
root = zarr.group(store)
root_eve = zarr.group(store_eve)
print(root.tree())
print(root_eve.tree())



Exploring the data

In [ ]:
# How to extract "keys" in the tree. OBS: the for loop is necessary to get the name. Calling for the keys (i.e., just root.keys()) does not provide the 
# name itself (in this case, "2010") like it would for a dictionary object
aia_tree_keys = root.keys()

for name in aia_tree_keys:
  print(type(name))
  print(name)

eve_tree_keys = root_eve.keys()

for name_eve in eve_tree_keys:
  print(type(name_eve))
  print(name_eve)

In [ ]:
# extract "values" name for AIA
tree_leaves = root[name]

for leaf in tree_leaves:
  print(leaf)
print("leaves are type: ", type(leaf))

# extract "values" name for EVE
tree_leaves_eve = root_eve[name_eve]

for leaf in tree_leaves_eve:
  print(leaf)
print("leaves are type: ", type(leaf))

In [ ]:
eve_channel = root_eve['MEGS-A']['C III']
eve_channel.info

In [ ]:
# selecting a specific channel
my_channel =  root['2010']['171A']
my_channel.info

In [ ]:
# the stored data themselves are numpy ndarrays...
print("type of stored data: ", type(my_channel[0]))
my_channel.shape

# ...but the data object itself is still a zarr array
print("type of object data: ", type(my_channel))


In [ ]:
# dask is used to convert the data-zarr object ("zarr.core.Array") to a data-dask object ("dask.array.core.Array")
all_image = da.from_array(my_channel)
print(type(my_channel))
all_image

In [ ]:
# extracting one eve channel and transforming to dask array
eve_C_III = da.from_array(eve_channel)
print(type(eve_C_III))
eve_C_III

In [ ]:
# selecting an image
image=all_image[0,:,:]

# plotting the image
plt.figure(figsize=(3,3))
plt.imshow(image,origin='lower',vmin=0,vmax=1000)

In [ ]:
# OBS: .attrs can be applied to a zarr array, but not to the group (i.e., if we do root.attrs it does not
# return the keys)
sorted(my_channel.attrs)

In [ ]:
# selecting data based on exposure and observation time
exptime = np.array(my_channel.attrs["EXPTIME"]) # this is provided in seconds
t_obs = np.array(my_channel.attrs["T_OBS"])


In [ ]:
# extracting dates from eve time
t_obs_eve = np.array(da.from_array(root_eve['MEGS-A']['Time']))
t_obs_eve

In [ ]:
plt.figure(figsize=(12,3))
plt.plot(pd.to_datetime(pd.Series(t_obs_eve[:300])),eve_C_III[:300]) # transforming to dataframe makes the dates axis in the plot nicer

In [ ]:
from matplotlib.ticker import StrMethodFormatter

plt.plot(exptime)
plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:,.5f}')) # 5 decimal places

In [ ]:
# select indices where the exposure time is less than 2 seconds
index = np.where(exptime < 2.0)
selected_images = da.from_array(my_channel)[index[0], :, :]
selected_images

In [ ]:
# select images based on their index
start_idx = 100
end_idx = 200
selected_images = da.from_array(my_channel)[start_idx:end_idx, :, :]
selected_images

In [ ]:
# downsampling the data to 256 by sampling every other pixel (memory reduction is num_pix^2)
num_pix = 2
sub_index = np.arange(0, 512, 2)

sub_image = da.from_array(my_channel)[start_idx:end_idx, sub_index, :]
sub_image = sub_image[:, :, sub_index]
sub_image

In [ ]:
# downsampling in time
df_time=pd.DataFrame(t_obs,index=np.arange(np.shape(t_obs)[0]),columns=['Time'])
df_time['Time'] = pd.to_datetime(df_time['Time'])
# select times at a frequency of 30 minutes (T is for minutes in pandas datetime object)
selected_times = pd.date_range(start='2010-09-09 00:00:00', end='2010-10-31 23:59:59', freq='60T',tz='UTC')
selected_times

In [ ]:
# The dataset (at root is different from the one in the integration-nuggets notebook)
selected_index = []
for i in selected_times:
    selected_index.append(np.argmin(abs(df_time["Time"] - i)))


#mark the missing timestamps in the data (we wouldn't have any, I believe these data are already reduced)
##missing_index=np.where(abs(df_time['Time'][selected_index]-selected_times)>pd.Timedelta('3m'))[0].tolist()
##for i in missing_index:
#    selected_index[i]=-1
    
time_index = list(filter(lambda x: x > 0, selected_index))
selected_image=all_image[time_index,:,:]
selected_image

Experiments on temporal alignment AIA-EVE

In [ ]:
# Transforming eve dates to datetime object
df_time_eve = pd.DataFrame(t_obs_eve,index=np.arange(np.shape(t_obs_eve)[0]),columns=['Time'])
df_time_eve['Time'] = pd.to_datetime(df_time_eve['Time'])


In [ ]:
print(df_time['Time'][0])
print(df_time_eve['Time'][0])

# it appears that the aia time series is time-zone-aware, while the eve one is not. Either remove or add time zone
df_time_aia = df_time['Time'].dt.tz_localize(None)

# this is definitely inefficient. The idea here is to downsample the eve time series. The loop below does what I have in mind, that is "for each
# date in aia, get the index of the closer one in the eve series". Once we have the list, we can extract the desired values from the eve data. 
# The idea here is to do this operation in a dedicated method that is called within the ZarrIrradianceDataModule class to ensure the required 1 
# to 1 mapping (1 aia example is associated with a single eve example)
select_eve_idx = []
for i in range(len(df_time_aia)):
    select_eve_idx.append(np.argmin(abs(df_time_eve['Time'] - df_time_aia[i])))

In [ ]:
# get data from path
aia_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_small.zarr')
eve_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_eve.zarr')

# get zarr group
root_aia = zarr.group(aia_data)
root_eve = zarr.group(eve_data)

# select data from zarr
aia_channel = root_aia['2010']['171A']
eve_channel = root_eve['MEGS-A']['C III']
# convert data to dask
aia_dask_image = da.from_array(aia_channel)[0,:,:]
eve_dask = da.from_array(eve_channel)

#image = aia_dask[0,:,:]
plt.figure(figsize=(3,3))
plt.imshow(aia_dask_image,origin='lower',vmin=0,vmax=1000)

Putting all together

In [ ]:
# get data from path
aia_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_small.zarr')
eve_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_eve.zarr')

# get zarr group
root_aia = zarr.group(aia_data)
root_eve = zarr.group(eve_data)

# select data from zarr
aia_channel = root_aia['2010']['171A']
eve_channel = root_eve['MEGS-A']['C III']
# convert data to dask
aia_dask = da.from_array(aia_channel)
eve_dask = da.from_array(eve_channel)

# get observation time
t_obs_aia_channel = aia_channel.attrs['T_OBS'] 
t_obs_eve_channel = np.array(da.from_array(root_eve['MEGS-A']['Time'])).tolist()

# transform to DataFrame
# AIA
df_t_obs_aia = pd.DataFrame(t_obs_aia_channel,index=np.arange(np.shape(t_obs_aia_channel)[0]),columns=['Time'])
df_t_obs_aia['Time'] = pd.to_datetime(df_t_obs_aia['Time']).dt.tz_localize(None)
df_t_obs_aia = df_t_obs_aia.set_index('Time')
df_t_obs_aia = df_t_obs_aia.sort_index().reset_index()
aia_year = int([k for k in root_aia.keys()][0])

# EVE
df_t_obs_eve = pd.DataFrame(t_obs_eve_channel,index=np.arange(np.shape(t_obs_eve_channel)[0]),columns=['Time'])
df_t_obs_eve['Time'] = pd.to_datetime(df_t_obs_eve['Time']).dt.tz_localize(None)
df_t_obs_eve = df_t_obs_eve.set_index('Time')
df_t_obs_eve = df_t_obs_eve.sort_index().reset_index()
df_t_obs_eve = df_t_obs_eve[df_t_obs_eve['Time'].dt.year == aia_year]

# slect a time range
selected_times = pd.date_range(start=df_t_obs_aia.iloc[0]['Time'], end=df_t_obs_aia.iloc[-1]['Time'], freq='60T',tz=None)
selected_index_aia = []
selected_index_eve = []
for i in range(len(selected_times)-1):
    selected_index_aia.append(np.argmin(abs(df_t_obs_aia['Time'] - selected_times[i] )))
    selected_index_eve.append(np.argmin(abs(df_t_obs_eve['Time'] - selected_times[i] )))

# get common times
selected_time_aia = df_t_obs_aia.iloc[selected_index_aia]
selected_time_eve = df_t_obs_eve.iloc[selected_index_eve]

# downsample AIA

Putting all together v2

In [ ]:
# get data from path
aia_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_small.zarr')
eve_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_eve.zarr')

# get zarr group
root_aia = zarr.group(aia_data)
root_eve = zarr.group(eve_data)

# select data from zarr
aia_channel = root_aia['2010']['171A']
eve_channel = root_eve['MEGS-A']['C III']

# get observation time
t_obs_aia_channel = aia_channel.attrs['T_OBS'] 
t_obs_eve_channel = np.array(da.from_array(root_eve['MEGS-A']['Time'])).tolist()

# transform to DataFrame
# AIA
df_t_aia =pd.DataFrame({'Time': pd.to_datetime(t_obs_aia_channel), 'Index_aia': np.arange(0,len(t_obs_aia_channel))})
df_t_aia['Time'] = pd.to_datetime(df_t_aia['Time']).dt.tz_localize(None) # this is needed for timezone-naive type
df_t_aia['Time'] = df_t_aia['Time'].dt.round('5min') # the cadence will be a user input
df_t_obs_aia = df_t_aia.drop_duplicates(subset='Time', keep='first') # removing potential duplicates derived by rounding
df_t_obs_aia.set_index('Time', inplace = True) # seting time as index

# EVE
df_t_eve =pd.DataFrame({'Time': pd.to_datetime(t_obs_eve_channel), 'Index_eve': np.arange(0,len(t_obs_eve_channel))})
df_t_eve['Time'] = pd.to_datetime(df_t_eve['Time'])  
df_t_eve['Time'] = df_t_eve['Time'].dt.round('5min')
df_t_obs_eve = df_t_eve.drop_duplicates(subset='Time', keep='first')
df_t_obs_eve.set_index('Time', inplace = True)

# the join function makes the intersection by index (which is now the time) and get the aligned indexes. The alternative would have involved
# sorting the dates, but we would have a mismatch with the original zarr
join_series = df_t_obs_aia.join(df_t_obs_eve, how='inner')

join_series


In [ ]:
# printing paired value to confirm they are matching
print("AIA date: ", df_t_aia.iloc[6130])
print("EVE date: ", df_t_eve.iloc[156928])

In [ ]:
# ignore this
#a = np.arange(1,10)
#b = np.arange(1,20)
#np.random.shuffle(b)
#aS = pd.Series(a)
#bS = pd.Series(b)
#idx = bS[bS.isin(aS)].index
#idxa = aS[aS.isin(bS)].index


Custom DataSet Class

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch

In [ ]:
# get data from path
aia_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_small.zarr')
eve_data = zarr.DirectoryStore('/mnt/sdomlv2_small/sdomlv2_eve.zarr')

# get zarr group
root_aia = zarr.group(aia_data)
root_eve = zarr.group(eve_data)

# select data from zarr
aia_channel = root_aia['2010']['171A']
eve_channel = root_eve['MEGS-A']['C III']
# convert data to dask
aia_dask_image = np.da.from_array(aia_channel)[0,:,:]
eve_dask = da.from_array(eve_channel)

#image = aia_dask[0,:,:]
plt.figure(figsize=(3,3))
plt.imshow(aia_dask_image,origin='lower',vmin=0,vmax=1000)

In [ ]:
class ZarrIrradianceDataset(Dataset):

    def __init__(self, aia_zarr_path, eve_zarr_path, wavelengths, ions, freq, months, transformations=None):

        """
        aia_zarr_path --> path: path to zarr aia data
        eve_zarr_path --> path: path to zarr eve data
        wavelengths   --> list: list of channels for aia
        ions          --> list: list of ions for eve
        freq          --> str: cadence used for rounding time series
        transformation: to be applied to aia in theory, but can stay None here

        """
        self.aia_zarr_path = aia_zarr_path
        self.eve_zarr_path = eve_zarr_path
        self.wavelengths = wavelengths
        self.ions = ions
        self.cadence = freq
        self.months = months
        self.transformations = transformations

        # get data from path
        self.aia_data = zarr.group(zarr.DirectoryStore(self.aia_zarr_path))
        self.eve_data = zarr.group(zarr.DirectoryStore(self.eve_zarr_path))

        # Temporal alignment of aia and eve data
        self.aligndata = self.__aligntime__()

        self.aligndata = self.aligndata.loc[self.aligndata.index.month.isin(self.months),:]
         
    def __len__(self):
        return self.aligndata.shape[0]
    
    
    def __getitem__(self, idx):

        index_row = self.aligndata.iloc[idx,:]

        aia_image_list = []

        for wavelength in self.wavelengths:

            # select data from zarr
            aia_channel = self.aia_data[index_row[f'year_{wavelength}']][wavelength]
            
            # convert data to dask
            aia_image_list.append(torch.tensor(np.array(da.from_array(aia_channel)[index_row[f'index_{wavelength}'],:,:])))

        euv_images = torch.stack(aia_image_list)
    
                   
        if self.transformations is not None:
            # transform as RGB + y to use transformations
            euv_images = euv_images.transpose() # this is probably needed for the dimension of the file they are passing
            # transformed = self.transformations(image=euv_images[..., :3], y=euv_images[..., 3:])
            # euv_images = torch.cat([transformed['image'], transformed['y']], dim=0)
            transformed = self.transformations(image=euv_images) # this applies transformations (here it just "toTensor", but 
                                                                # it could be like rotating, change colors etc..)
            euv_images = transformed['image']
        

        eve_ion_list = []

        for ion in self.ions:
            eve_ion_list.append(torch.tensor(np.array(da.from_array(self.eve_data['MEGS-A'][ion])[index_row['index_eve']])))
        

        eve_data = torch.stack(eve_ion_list)

        return euv_images, eve_data

    
    def __aligntime__(self):

        """

        This function extracts the common indexes across aia and eve datasets, considering potential missing values.

        """

        # select data from zarr
        for i,wavelength in enumerate(self.wavelengths):

            for j,key in enumerate(self.aia_data.keys()):

                aia_channel = self.aia_data[key][wavelength]

                # get observation time
                t_obs_aia_channel = aia_channel.attrs['T_OBS'] 

                if j == 0:

                    # transform to DataFrame
                    # AIA
                    df_t_aia =pd.DataFrame({'Time': pd.to_datetime(t_obs_aia_channel),f'index_{wavelength}': np.arange(0,len(t_obs_aia_channel))})
                    df_t_aia[f'year_{wavelength}'] = key

                else:

                    df_tmp_aia =pd.DataFrame({'Time': pd.to_datetime(t_obs_aia_channel), f'index_{wavelength}': np.arange(0,len(t_obs_aia_channel))})
                    df_tmp_aia[f'year_{wavelength}'] = key

                    df_t_aia = pd.concat([df_t_aia,df_tmp_aia], ignore_index = True)

            df_t_aia['Time'] = pd.to_datetime(df_t_aia['Time']).dt.tz_localize(None) # this is needed for timezone-naive type
            df_t_aia['Time'] = df_t_aia['Time'].dt.round(self.cadence) 
            df_t_obs_aia = df_t_aia.drop_duplicates(subset='Time', keep='first') # removing potential duplicates derived by rounding
            df_t_obs_aia.set_index('Time', inplace = True)
            
            if i == 0:
                join_series = df_t_obs_aia
            else:
                join_series = join_series.join(df_t_obs_aia, how='inner')

        # EVE
        t_obs_eve_channel = np.array(da.from_array(self.eve_data['MEGS-A']['Time'])).tolist()
        df_t_eve =pd.DataFrame({'Time': pd.to_datetime(t_obs_eve_channel), 'index_eve': np.arange(0,len(t_obs_eve_channel))})
        df_t_eve['Time'] = pd.to_datetime(df_t_eve['Time'])  
        df_t_eve['Time'] = df_t_eve['Time'].dt.round(self.cadence)
        df_t_obs_eve = df_t_eve.drop_duplicates(subset='Time', keep='first')
        df_t_obs_eve.set_index('Time', inplace = True)
        join_series = join_series.join(df_t_obs_eve, how='inner')

        # remove missing eve data (missing values are labeled with negative values)
        for ion in self.ions:

            ion_data = np.array(da.from_array(self.eve_data['MEGS-A'][ion]))

            join_series = join_series.loc[ion_data[join_series['index_eve']] > 0,:]

        return join_series


In [ ]:
mytab = dataset.aligndata
day_list = [17,28]
mytab.loc[mytab.index.day.isin(day_list),:]

In [ ]:
#dataset = ZarrIrradianceDataset('/mnt/sdomlv2_full/sdomlv2.zarr', '/mnt/sdomlv2_full/sdomlv2_eve.zarr', ['171A','304A','193A'], ['Fe IX','Fe VIII'], '10min', [1,3], transformations=None)
dataset = ZarrIrradianceDataset('/mnt/tmp/sdomlv2.zarr', '/mnt/tmp/sdomlv2_eve.zarr', ['171A','304A','193A'], ['Fe IX','Fe VIII'], '10min', [1,3], transformations=None)
#mytable = dataset.__aligntime__()
#im_stack,eve_stack = dataset[0]
#print(eve_stack)
dataset.aligndata

In [ ]:
join_table = dataset.aligndata
reduced_table = join_table.loc[(join_table.index.year == 2010) & (join_table.index.day == 28)]
reduced_table

In [ ]:
im_stack,eve_stack = dataset[0]
plt.imshow(dataset[0][2,:,:])

In [ ]:
# Dataset class for zarr format (we can finish it together..the idea is to follow the structure of the next cell and make minimal changes to integrate
# the zarr format). 

class ZarrIrradianceDataModule(pl.LightningDataModule):

    def __init__(self, path_2_aia, path_2_eve, wavelengths, ions, frequency, batch_size: int = 32, num_workers=None,
                 train_transforms=None, val_transforms=None, val_months=[10,1], test_months=[11,12], holdout_months=None):
        """ Loads paired data samples of AIA EUV images and EVE irradiance measures.

        Note: Input data needs to be paired.
        Parameters
        ----------
        path_2_zarr: path to the matches
        path_2_eve: path to the EVE data file
        batch_size: batch size (default is 32)
        num_workers: number of workers (needed for the training)
        train_transforms: transformations to be applied on the training set
        val_transforms: transformations to be applied on the validation set
        val_months: 
        """

        super().__init__()
        self.num_workers = num_workers if num_workers is not None else os.cpu_count() // 2
        self.path_2_aia = path_2_aia
        self.path_2_eve = path_2_eve
        self.batch_size = batch_size
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms
        self.wavelengths = wavelengths
        self.ions = ions
        self.cadence = frequency
        self.val_months = val_months
        self.test_months = test_months
        self.holdout_months = holdout_months

        self.months = [i for i in range(1, 13)]


    def setup(self, stage=None):

        train_months = [i for i in self.months if i not in self.test_months]
        self.train_months = [i for i in train_months if i not in self.val_months]

        self.train_ds = ZarrIrradianceDataset(self.path_2_aia, self.path_2_eve, 
                                              self.wavelengths, self.ions, self.cadence, self.train_months, self.train_transforms)
        self.test_ds = ZarrIrradianceDataset(self.path_2_aia, self.path_2_eve, 
                                              self.wavelengths, self.ions, self.cadence, self.test_months, self.test_transforms)
        self.valid_ds = ZarrIrradianceDataset(self.path_2_aia, self.path_2_eve, 
                                              self.wavelengths, self.ions, self.cadence, self.val_months, self.val_transforms)


Older classes

In [ ]:
# Coping this here for reference when writing the above class

class IrradianceDataModule(pl.LightningDataModule):

    def __init__(self, stacks_csv_path, eve_npy_path, wavelengths, batch_size: int = 32, num_workers=None,
                 train_transforms=None, val_transforms=None, val_months=[10, 1], test_months=[11,12], holdout_months=None):
        """ Loads paired data samples of AIA EUV images and EVE irradiance measures.

        Input data needs to be paired.
        Parameters
        ----------
        stacks_csv_path: path to the matches
        eve_npy_path: path to the EVE data file
        """
        super().__init__()
        self.num_workers = num_workers if num_workers is not None else os.cpu_count() // 2
        self.stacks_csv_path = stacks_csv_path
        self.eve_npy_path = eve_npy_path
        self.batch_size = batch_size
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms
        self.wavelengths = wavelengths
        self.val_months = val_months
        self.test_months = test_months
        self.holdout_months = holdout_months


        # get input dataframe to see how they actually look like --> transform zarr data to that format


    # TODO: Extract SDO dats for train/valid/test sets
    def setup(self, stage=None):
        # load data stacks (paired samples)
        # AIA images as csv file, eve data loaded as numpy array
        df = pd.read_csv(self.stacks_csv_path, parse_dates=['eve_dates'])
        eve_data = np.load(self.eve_npy_path)

        # make sure there is a 1 to 1 mapping
        print(len(eve_data), len(df))
        assert len(eve_data) == len(df), 'Inconsistent data state. EVE and AIA stacks do not match.'

        # select indeces associated with validaton and test set        
        test_cond_2014 = df.eve_dates.dt.year.isin([2014])
        val_condition = df.eve_dates.dt.month.isin(self.val_months)
        test_condition = df.eve_dates.dt.month.isin(self.test_months) | test_cond_2014 

        # not sure what they mean for holdout_months, but here it is taking all the indeces not belonging to the other sets and 
        # considering the remaining ones as training set indeces
        if self.holdout_months is not None:
            holdout_condition = df.eve_dates.dt.month.isin(self.holdout_months)
            train_condition = ~(val_condition | test_condition | holdout_condition)
        else:
            train_condition = ~(val_condition | test_condition)

        # actual splitting of the images in train,test and valid sets
        valid_df = df[val_condition]
        test_df = df[test_condition]
        train_df = df[train_condition]

        # call to Dataset class (which assumes csv/numpy array as input) --> this creates a Dataset object type, which can be managed through
        # the DataLoader class
        self.train_ds = IrradianceDataset(np.array(train_df.aia_stack), eve_data[train_df.index], 
                                          self.wavelengths, self.train_transforms)
        self.valid_ds = IrradianceDataset(np.array(valid_df.aia_stack), eve_data[valid_df.index], 
                                          self.wavelengths, self.val_transforms)
        self.test_ds = IrradianceDataset(np.array(test_df.aia_stack), eve_data[test_df.index], 
                                         self.wavelengths, self.val_transforms)

        # extracting dates associated with each set
        self.train_dates = train_df['eve_dates']
        self.valid_dates = valid_df['eve_dates']
        self.test_dates = test_df['eve_dates']

    # the DataLoader class makes the Dataset object an iterable, allowing to split into batches (useful for the training/testing loops)
    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=self.num_workers)

    def predict_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, num_workers=self.num_workers)
    

# Custom Dataset class (which inherits from the Dataset class from Pytorch)
class IrradianceDataset(Dataset):

    def __init__(self, euv_paths, eve_irradiance, wavelengths, transformations=None):
        """ Loads paired data samples of AIA EUV images and EVE irradiance measures.

        # OBS: it is run once when instantiating the Dataset object.

        Input data needs to be paired.
        Parameters
        ----------
        euv_paths: list of numpy paths as string (n_samples, )
        eve_irradiance: list of the EVE data values (n_samples, eve_channels)
        """
        assert len(euv_paths) == len(eve_irradiance), 'Input data needs to be paired!'
        self.euv_paths = euv_paths
        self.eve_irradiance = eve_irradiance
        self.transformations = transformations
        self.wavelengths = wavelengths

    # description: it returns the size of the dataset (number of images). Used by Dataloader
    def __len__(self):
        return len(self.euv_paths)
    
    # description: it  loads and returns a sample from the dataset at the given index
    def __getitem__(self, idx):
        euv_images = np.load(self.euv_paths[idx])[self.wavelengths, ...]
        eve_data = self.eve_irradiance[idx]
        if self.transformations is not None:
            # transform as RGB + y to use transformations
            euv_images = euv_images.transpose() # this is probably needed for the dimension of the file they are passing
            # transformed = self.transformations(image=euv_images[..., :3], y=euv_images[..., 3:])
            # euv_images = torch.cat([transformed['image'], transformed['y']], dim=0)
            transformed = self.transformations(image=euv_images) # this applies transformations (here it just "toTensor", but 
                                                                 # it could be like rotating, change colors etc..)
            euv_images = transformed['image']

        return euv_images, torch.tensor(eve_data, dtype=torch.float32)

